# FishSizeEstimate — Integrated Training, Stereo Reconstruction & Metric Evaluation (V7)

**Research notebook / end-to-end experimental pipeline.** Includes full ROS-image self-supervised training from V6, frame pairing, session-aware calibration, disparity/depth previews, an annotation template for fish endpoints, and **gated** metric evaluation. No fabricated fish-length predictions.

**Important**: Only 11 recovered recordings are present; nine Dataset A recordings belong to Fish 0, two Dataset B recordings contain multiple fish. Thus full-recording pretraining is feasible, but robust supervised centimetre prediction **cannot be claimed** without verified nose/tail correspondence, calibration units and independent fish identities. By default the notebook will fully train the visual encoder and produce *actual stereo frame previews / disparity diagnostics*; it will not output invented cm metrics.

**Input**: `/kaggle/input/datasets/wafabila/fishsizeestimate-dataset` · GPU T4 recommended · Approx. 23 GiB of input ROS bags. Save Version with all outputs after execution. Third-party dataset/model rights must be verified before republication. Never execute the supplied third-party `.pt` through untrusted pickle loading.


## 0 · Setup & reproducibility

**Default configuration intentionally attempts every decoded image from every `.bag`:** `FRAME_STRIDE=1`, `MAX_FRAMES_PER_BAG=None`, `MAX_TRAIN_STEPS=None`. It does **not** silently impose a tiny sample cap. Frame content is resized for model input, which is distinct from skipping frames. CPU/GPU capability and stream health are checked before training.

In [ ]:
import os, sys, json, time, random, math, gc, importlib.util, subprocess, platform
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

ROOT = Path('/kaggle/input/datasets/wafabila/fishsizeestimate-dataset')
OUT = Path('/kaggle/working/fishsizeestimate_integrated_v7')
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Dataset not mounted: {ROOT}'

SEED = 42
FRAME_STRIDE = 1                 # 1 = process each accessible image message
MAX_FRAMES_PER_BAG = None        # None = no per-bag image limit
MAX_TRAIN_STEPS = None          # None = no training-step limit
IMAGE_SIZE = 112                # resize, NOT sampling from recordings
BATCH_SIZE = 32
LR = 3e-4
WEIGHT_DECAY = 1e-4
TEMP = 0.2
LOG_EVERY_STEPS = 50
CHECKPOINT_EVERY_STEPS = 250
PREVIEW_MESSAGES = 30
STEREO_MAX_DELTA_MS = 50.0  # provisional QA tolerance; not proof of calibrated synchronization
RESUME_COMPLETED_BAGS = True

random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import cv2
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:',sys.version.split()[0],'Torch:',torch.__version__,'OpenCV:',cv2.__version__)
print('Device:', DEVICE, 'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Dataset:',ROOT)
print('Output:',OUT)
print('CONFIG:',{'FRAME_STRIDE':FRAME_STRIDE,'MAX_FRAMES_PER_BAG':MAX_FRAMES_PER_BAG,
                 'MAX_TRAIN_STEPS':MAX_TRAIN_STEPS,'IMAGE_SIZE':IMAGE_SIZE,'BATCH_SIZE':BATCH_SIZE})

if importlib.util.find_spec('rosbags') is None:
    print('Installing rosbags (only dependency not guaranteed in Kaggle)...')
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','rosbags>=0.10,<0.12'])
from rosbags.highlevel import AnyReader
print('[PASS] ROS reader available')

## 1 · Manifest, physical labels, recording coverage plan

The 9 physical-length labels are kept as immutable ground truth. A visual encoder does **not** use those cm values as its training target. We record the real original recording paths from `source_manifest.json`, so model/data lineage is reproducible.

In [ ]:
all_files = sorted(p for p in ROOT.rglob('*') if p.is_file())
bags = sorted((p for p in all_files if p.suffix.lower()=='.bag'), key=lambda p:p.name)
assert len(bags)==11, f'Expected 11 uploaded recordings, found {len(bags)}'
manifest_path = next(p for p in all_files if p.name=='source_manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf8'))
manifest_lookup = {r['upload_name']:r['original_path'] for r in manifest['files']}

def original_path(path):
    return manifest_lookup.get(path.name,path.name)

inventory=[]
for i,p in enumerate(bags):
    with p.open('rb') as fh: magic=fh.read(13)
    inventory.append({'bag_index':i,'name':p.name,'source_path':original_path(p),
                      'dataset':'B' if '/Dataset_B/' in original_path(p) else 'A',
                      'size_gib':round(p.stat().st_size/(1024**3),4),
                      'magic_valid':magic.startswith(b'#ROSBAG V2.0')})
inv = pd.DataFrame(inventory)
display(inv[['bag_index','dataset','source_path','size_gib','magic_valid']])
assert inv.magic_valid.all(), 'An uploaded ROS header was not recognized'
inv.to_csv(OUT/'input_bag_inventory.csv',index=False)

gt_path=next(p for p in all_files if p.name.endswith('fish_gt_measures.csv'))
gt=pd.read_csv(gt_path,dtype=str,encoding='utf-8-sig')
gt['Total_Length_cm']=pd.to_numeric(gt['Total_Length_cm'].str.replace(',','.',regex=False), errors='raise')
display(gt[['Dataset','Identifier','Total_Length_cm']])
print('[PROVENANCE] available ROS bags:',len(bags),'total GiB:',inv.size_gib.sum())
print('[LIMITATION] source manifest state:',manifest.get('status'))
print('[LIMITATION] physical fish identities:',len(gt),'but source frames are not 9 independent labels.')

## 2 · Decode ROS image messages (loss-aware)

Uses `rosbags.highlevel.AnyReader` to stream ROS1 messages and `cv2.imdecode` for `sensor_msgs/CompressedImage` or 8-bit `sensor_msgs/Image`. The model uses each **decoded** image message; non-image topics are not relevant. The script logs attempted, successful, failed, and consumed counts **per bag**, so partial or corrupt streams cannot silently masquerade as complete coverage.

**Important:** `rosbags` reading can fail on a damaged bag index or ROS chunk. We report such failures honestly, save progress, and continue to the next file.

In [ ]:
def is_image_topic(conn):
    typ=conn.msgtype.replace('/msg/','/')
    return typ.endswith('/CompressedImage') or typ.endswith('/Image')

def decode_image(message, msgtype):
    typ=msgtype.replace('/msg/','/')
    if typ.endswith('/CompressedImage'):
        encoded=np.frombuffer(message.data,dtype=np.uint8)
        bgr=cv2.imdecode(encoded,cv2.IMREAD_COLOR)
        return bgr
    if not typ.endswith('/Image'):
        return None
    enc=str(message.encoding).lower()
    h,w,step=int(message.height),int(message.width),int(message.step)
    if h<=0 or w<=0 or step<=0: return None
    raw=np.frombuffer(message.data,dtype=np.uint8)
    if raw.size < h*step:return None
    rows=raw[:h*step].reshape(h,step)
    if enc in ('bgr8','rgb8') and step>=3*w:
        img=rows[:,:3*w].reshape(h,w,3)
        return img.copy() if enc=='bgr8' else cv2.cvtColor(img,cv2.COLOR_RGB2BGR)
    if enc=='mono8' and step>=w:
        return cv2.cvtColor(rows[:,:w],cv2.COLOR_GRAY2BGR)
    return None

def resize_bgr(frame,size=IMAGE_SIZE):
    return cv2.resize(frame,(size,size),interpolation=cv2.INTER_AREA if max(frame.shape[:2])>=size else cv2.INTER_LINEAR)

# Quick actual preview from first file; no claim this covers the complete recording.
from PIL import Image
preview=[]
with AnyReader([bags[0]]) as reader:
    topics=pd.DataFrame([{'topic':c.topic,'type':c.msgtype,'count':c.msgcount} for c in reader.connections])
    print('Topics in first bag:')
    display(topics)
    image_connections=[c for c in reader.connections if is_image_topic(c)]
    for j,(conn,t,blob) in enumerate(reader.messages(connections=image_connections)):
        if j>=PREVIEW_MESSAGES or len(preview)>=4:break
        try:
            msg=reader.deserialize(blob,conn.msgtype)
            img=decode_image(msg,conn.msgtype)
            if img is not None: preview.append((conn.topic,cv2.cvtColor(resize_bgr(img,224),cv2.COLOR_BGR2RGB)))
        except Exception as exc:
            print('Preview decode warning:',str(exc)[:180])
assert preview, 'No decodable image found in first ROS bag; inspect topic definitions before training.'
fig,axs=plt.subplots(1,len(preview),figsize=(4*len(preview),4))
for ax,(topic,image) in zip(np.atleast_1d(axs),preview):
    ax.imshow(image);ax.set_title(topic[:45],fontsize=8);ax.axis('off')
plt.tight_layout();plt.savefig(OUT/'actual_ros_preview.png',dpi=120);plt.show()
print('[PASS] Actual ROS frames decoded from source data (not specimen reference photos).')

## 3 · Model and learning objective

An original compact 4-stage CNN encoder is trained from random initialization. Two stochastic augmentations of the **same actual recording frame** are a positive pair. Other images in the minibatch serve as negatives. This is **self-supervised contrastive representation learning**. Loss decreasing is an optimization diagnostic, **not physical-length accuracy**.

For strict reproducibility, there is no preloaded `.pt` checkpoint or unsafe deserialization. Source checkpoint `generic_fish_segmenter.pt` is intentionally not executed.

In [ ]:
class ConvEncoder(nn.Module):
    def __init__(self,embedding_dim=128):
        super().__init__()
        layers=[]
        in_ch=3
        for out_ch in (32,64,96,128):
            layers.extend([nn.Conv2d(in_ch,out_ch,3,stride=2,padding=1,bias=False),
                           nn.BatchNorm2d(out_ch),nn.SiLU(inplace=True),
                           nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False),
                           nn.BatchNorm2d(out_ch),nn.SiLU(inplace=True)])
            in_ch=out_ch
        self.backbone=nn.Sequential(*layers,nn.AdaptiveAvgPool2d((1,1)),nn.Flatten())
        self.embedding=nn.Linear(128,embedding_dim)
        self.projector=nn.Sequential(nn.Linear(embedding_dim,256),nn.ReLU(inplace=True),nn.Linear(256,128))
    def forward(self,x):
        features=F.normalize(self.embedding(self.backbone(x)),dim=-1)
        projection=F.normalize(self.projector(features),dim=-1)
        return features,projection

def augment_batch(images):
    # images: CPU uint8 B,H,W,3 RGB, converted to device (no image file cache)
    x=torch.from_numpy(np.stack(images)).to(DEVICE,non_blocking=True).permute(0,3,1,2).float()/255.
    # Independent random brightness/contrast and horizontal mirror per example.
    B=x.shape[0]
    mirror=(torch.rand(B,device=DEVICE)<0.5)
    x[mirror]=torch.flip(x[mirror],[-1])
    brightness=torch.empty(B,1,1,1,device=DEVICE).uniform_(0.8,1.2)
    contrast=torch.empty(B,1,1,1,device=DEVICE).uniform_(0.8,1.2)
    x=(x-0.5)*contrast+0.5
    x=(x*brightness).clamp(0,1)
    if B>0:
        x=(x+0.02*torch.randn_like(x)).clamp(0,1)
    return x

def info_nce(z1, z2, temperature=TEMP):
    """Stable NT-Xent: float32 logits even under CUDA autocast(float16)."""
    assert z1.ndim == 2 and z1.shape == z2.shape and z1.shape[0] >= 2
    # Explicitly disable autocast here: float16 cannot represent -1e9.
    with torch.autocast(device_type=DEVICE.type, enabled=False):
        z = F.normalize(torch.cat([z1.float(), z2.float()], dim=0), dim=1)
        logits = torch.matmul(z, z.T) / float(temperature)
        logits.fill_diagonal_(-1e4)  # Safe in both float32 and float16.
        half = z1.shape[0]
        labels = (torch.arange(2 * half, device=z.device) + half) % (2 * half)
        loss = F.cross_entropy(logits, labels)
    return loss

model=ConvEncoder().to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
try:
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
except TypeError:
    scaler=torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')
print('Trainable parameters:',sum(p.numel() for p in model.parameters()))
print('Learning task: contrastive image embeddings, NOT fish length in centimetres')

## 4 · Checkpoint/resume protocol

Save actual model, optimizer, scaler and Python/NumPy/PyTorch RNG states as a **local trusted checkpoint** (we only load our own files). Each finished bag is marked `completed` in a progress ledger. If a session stops inside one bag, restarting will revisit the current bag; prior completed bag runs are not repeated.

**Use a fresh Kaggle session carefully:** Files in `/kaggle/working` persist only if the notebook version/output is saved; resuming across sessions requires attaching the previous output as input and restoring files manually. This notebook does not claim background execution.

In [ ]:
PROGRESS=OUT/'progress.json'
CKPT=OUT/'latest_checkpoint.pt'
LOG=OUT/'train_steps.csv'

state={'completed_bags':[],'bag_stats':{},'global_step':0,'total_decoded':0,'total_consumed':0,
       'warnings':[],'source_bags':len(bags),'config':{
          'frame_stride':FRAME_STRIDE,'max_frames_per_bag':MAX_FRAMES_PER_BAG,
          'max_train_steps':MAX_TRAIN_STEPS,'image_size':IMAGE_SIZE,'batch_size':BATCH_SIZE, 'stereo_max_delta_ms':STEREO_MAX_DELTA_MS}}
loss_history=[]

def write_json_atomic(path,obj):
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
    temp.replace(path)

def checkpoint():
    temp=CKPT.with_suffix('.pt.tmp')
    torch.save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),
                'scaler':scaler.state_dict(),'step':state['global_step'],
                'completed_bags':state['completed_bags'],
                'torch_rng':torch.get_rng_state(),
                'numpy_rng':np.random.get_state(),
                'python_rng':random.getstate(),
                'config':state['config']},temp)
    temp.replace(CKPT)
    write_json_atomic(PROGRESS,state)

if RESUME_COMPLETED_BAGS and CKPT.exists() and PROGRESS.exists():
    checkpoint_data=torch.load(CKPT,map_location=DEVICE,weights_only=False) # trusted self-generated local file only
    existing=json.loads(PROGRESS.read_text(encoding='utf-8'))
    assert existing.get('config')==state['config'], 'Configuration differs from saved checkpoint; use a separate output directory'
    model.load_state_dict(checkpoint_data['model'])
    optimizer.load_state_dict(checkpoint_data['optimizer'])
    scaler.load_state_dict(checkpoint_data['scaler'])
    state=existing
    torch.set_rng_state(checkpoint_data['torch_rng'].cpu())
    np.random.set_state(checkpoint_data['numpy_rng'])
    random.setstate(checkpoint_data['python_rng'])
    print('[RESUME] completed bags:',len(state['completed_bags']),'global steps:',state['global_step'])
else:
    print('[FRESH] training weights initialized; no prior checkpoint')

if LOG.exists():
    loss_history=pd.read_csv(LOG).to_dict('records')

def save_log():
    if loss_history:
        pd.DataFrame(loss_history).to_csv(LOG,index=False)

## 5 · FULL streaming training over every available bag

**The actual training loop** processes each readable ROS image message once, creates two stochastic augmented views, runs forward/backward optimizer steps, and records exact coverage. `MAX_* = None` and `FRAME_STRIDE=1` mean no artificial sampling cap. The only excluded content is unsupported image encodings, damaged messages, and ROS non-image topics.

ROS bag iteration may take substantial time; an earlier audit ran quickly only because it checked headers and tables. A missing or truncated Dataset B file cannot be trained on. Check GPU and session limits before saving a version.

Train will not claim complete coverage if a recording aborts mid-stream. If any bag errors, the summary explicitly reports it.

**V5 safety:** GPU mixed-precision preflight is run before any full-bag streaming. Training-related numeric/CUDA errors halt early rather than incorrectly marking all bags as corrupt.


In [ ]:
assert FRAME_STRIDE>=1 and BATCH_SIZE>=2
model.train()
start_all=time.monotonic()

# Avoid holding more than one batch of decoded frames in memory.
class TrainingStepError(RuntimeError):
    """Distinguish model failures from corrupted ROS files."""


def train_on_batch(image_batch, bag_name):
    if len(image_batch)<2:
        return None
    model.train()
    optimizer.zero_grad(set_to_none=True)
    x1=augment_batch(image_batch)
    x2=augment_batch(image_batch)
    if DEVICE.type=='cuda':
        with torch.autocast(device_type='cuda',dtype=torch.float16):
            _,z1=model(x1); _,z2=model(x2)
            loss=info_nce(z1.float(),z2.float())
    else:
        _,z1=model(x1); _,z2=model(x2)
        loss=info_nce(z1,z2)
    if not torch.isfinite(loss):
        raise FloatingPointError(f'Non-finite training loss from {bag_name}')
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(),2.0)
    scaler.step(optimizer)
    scaler.update()
    state['global_step']+=1
    val=float(loss.detach().cpu())
    loss_history.append({'step':state['global_step'],'loss':val,'bag':bag_name,'batch_size':len(image_batch)})
    return val

# PRE-FLIGHT: run real model+loss+backward BEFORE reading all 22.8 GiB.
# Tiny synthetic frames are for code validation only; they are NOT training data.
print("[PREFLIGHT] Checking CUDA autocast, loss and gradients...")
model.train()
optimizer.zero_grad(set_to_none=True)
try:
    smoke_images = [np.random.randint(0, 256, (IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
                    for _ in range(4)]
    a = augment_batch(smoke_images)
    b = augment_batch(smoke_images)
    if DEVICE.type == 'cuda':
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            _, u = model(a)
            _, v = model(b)
            smoke_loss = info_nce(u, v)
    else:
        _, u = model(a)
        _, v = model(b)
        smoke_loss = info_nce(u, v)
    assert smoke_loss.dtype == torch.float32, f'Loss dtype is {smoke_loss.dtype}'
    assert torch.isfinite(smoke_loss).item(), 'Smoke-test loss is not finite'
    smoke_loss.backward()
    assert any(p.grad is not None and torch.isfinite(p.grad).all().item()
               for p in model.parameters()), 'No finite gradients'
    print(f'[PREFLIGHT PASS] loss={smoke_loss.item():.4f}; dtype={smoke_loss.dtype}; gradients finite')
finally:
    optimizer.zero_grad(set_to_none=True)
    del a, b, u, v, smoke_images
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

for bag in bags:
    if bag.name in state['completed_bags']:
        print('[SKIP COMPLETED]',bag.name)
        continue
    print('\n'+'='*70+'\n[START BAG]',bag.name)
    count_messages=0; count_decoded=0; count_skipped=0; count_failed=0; count_trained=0
    image_batch=[]; bag_status='complete'; bag_error=None
    begin=time.monotonic()
    topics_record=[]
    camera_times={'left': [], 'right': []}
    camera_counts={'left': 0, 'right': 0, 'other': 0}
    try:
        with AnyReader([bag]) as reader:
            connections=[c for c in reader.connections if is_image_topic(c)]
            topics_record=[{'topic':c.topic,'type':c.msgtype,'expected':int(c.msgcount)} for c in connections]
            if not connections:
                raise RuntimeError('No supported image topics in ROS metadata')
            for connection,ts,raw in reader.messages(connections=connections):
                count_messages+=1
                # Collect actual image-message timestamps; no frame data retained.
                topic_lower = connection.topic.lower()
                side = ('left' if '/left/' in topic_lower else
                        'right' if '/right/' in topic_lower else 'other')
                camera_counts[side] += 1
                if side in camera_times:
                    camera_times[side].append(int(ts))
                if (count_messages-1)%FRAME_STRIDE:
                    count_skipped+=1; continue
                if MAX_FRAMES_PER_BAG is not None and count_decoded>=MAX_FRAMES_PER_BAG:
                    bag_status='capped_by_user_config';break
                if MAX_TRAIN_STEPS is not None and state['global_step']>=MAX_TRAIN_STEPS:
                    bag_status='capped_by_user_config';break
                try:
                    msg=reader.deserialize(raw,connection.msgtype)
                    bgr=decode_image(msg,connection.msgtype)
                    if bgr is None:count_failed+=1;continue
                    resized=resize_bgr(bgr)
                    image_batch.append(cv2.cvtColor(resized,cv2.COLOR_BGR2RGB))
                    count_decoded+=1
                except Exception as ex:
                    count_failed+=1
                    if count_failed<=3:print('[DECODE WARNING]',type(ex).__name__,str(ex)[:160])
                    continue
                if len(image_batch)>=BATCH_SIZE:
                    loss=train_on_batch(image_batch,bag.name)
                    count_trained+=len(image_batch);image_batch.clear()
                    if state['global_step']%LOG_EVERY_STEPS==0:
                        print(f' step={state["global_step"]:,} decoded={count_decoded:,} trained={count_trained:,} loss={loss:.4f} elapsed_s={time.monotonic()-begin:.0f}',flush=True)
                        save_log()
                    if state['global_step']%CHECKPOINT_EVERY_STEPS==0:
                        # Mid-bag checkpoint does not mark this bag completed.
                        checkpoint()
            if len(image_batch)>=2 and (MAX_TRAIN_STEPS is None or state['global_step']<MAX_TRAIN_STEPS):
                loss=train_on_batch(image_batch,bag.name)
                count_trained+=len(image_batch)
            elif len(image_batch)==1:
                count_skipped+=1
            image_batch.clear()
    except Exception as ex:
        bag_status='error'
        bag_error=f'{type(ex).__name__}: {str(ex)[:400]}'
        print('[BAG FAILED]',bag_error)
        image_batch.clear()
        fatal_train_error = isinstance(ex, (RuntimeError, FloatingPointError)) and (
            'overflow' in str(ex).lower() or 'cuda' in str(ex).lower() or
            'out of memory' in str(ex).lower() or 'loss' in str(ex).lower())
    stats={'status':bag_status,'message_events_seen':count_messages,
           'decoded_images':count_decoded,'trained_images':count_trained,
           'skipped_or_remainder':count_skipped,'image_decode_failures':count_failed,
           'duration_s':round(time.monotonic()-begin,2),'topics':topics_record,
           'camera_image_counts':camera_counts, 'camera_timestamps_ns':camera_times, 'error':bag_error}
    state['bag_stats'][bag.name]=stats
    # Completed means the iterator exhausted, not simply that one training step ran.
    if bag_status=='complete':state['completed_bags'].append(bag.name)
    else:state['warnings'].append({'bag':bag.name,'status':bag_status,'error':bag_error})
    state['total_decoded']=sum(x['decoded_images'] for x in state['bag_stats'].values())
    state['total_consumed']=sum(x['trained_images'] for x in state['bag_stats'].values())
    checkpoint();save_log()
    if bag_status == 'error' and fatal_train_error:
        raise RuntimeError(f'TRAINING FAILED on {bag.name}: {bag_error}. Stopped early to avoid wasting time.')
    print('[FINISHED BAG]',bag_status,'decoded=',count_decoded,'trained=',count_trained,
          'errors=',count_failed,'elapsed_s=',stats['duration_s'],flush=True)
    if MAX_TRAIN_STEPS is not None and state['global_step']>=MAX_TRAIN_STEPS:
        print('[STOP] User-configured MAX_TRAIN_STEPS reached');break

print('\n[TRAINING LOOP DONE] elapsed minutes:',round((time.monotonic()-start_all)/60,1))
print('Completed bags:',len(state['completed_bags']),'/',len(bags))
print('Successfully decoded images:',state['total_decoded'])
print('Images used in optimizer batches:',state['total_consumed'])
print('Optimizer steps:',state['global_step'])
assert state['global_step']>0, 'No optimizer step completed. Inspect ROS topics and decode errors.'

## 6 · Coverage report, training curve & checkpoint export

Use the actual recorded counts. A bag is counted as **complete** only if the ROS message iterator finishes without failure or user cap. Failed decoded messages remain visible. Completion of all eleven files is necessary to claim *all available files have been fully traversed*, not necessarily every message decoded successfully. No centimetre estimates or fake accuracy are generated.

In [ ]:
rows=[]
for p in bags:
    s=state['bag_stats'].get(p.name,{})
    rows.append({'bag':p.name,'dataset':'B' if '/Dataset_B/' in original_path(p) else 'A',
                 'status':s.get('status','not_attempted'),
                 'image_events_seen':s.get('message_events_seen',0),
                 'decoded_images':s.get('decoded_images',0),
                 'trained_images':s.get('trained_images',0),
                 'decode_failures':s.get('image_decode_failures',0),
                 'left_images':s.get('camera_image_counts',{}).get('left',0),
                 'right_images':s.get('camera_image_counts',{}).get('right',0),
                 'duration_s':s.get('duration_s',0),'error':s.get('error')})
coverage=pd.DataFrame(rows)
coverage.to_csv(OUT/'all_bag_coverage.csv',index=False)
display(coverage)
complete_count=int((coverage.status=='complete').sum())
print('COMPLETE STREAMS:',complete_count,'/',len(bags))
print('DECODE FAILURES:',int(coverage.decode_failures.sum()))
print('TRAINED IMAGES:',int(coverage.trained_images.sum()))
if complete_count!=len(bags):
    print('[WARNING] Some files were not fully processed. Do NOT report full-dataset training as finished.')
if int(coverage.decode_failures.sum())>0:
    print('[WARNING] Some ROS image messages could not be decoded.')

if loss_history:
    history=pd.DataFrame(loss_history)
    fig,ax=plt.subplots(figsize=(12,4))
    ax.plot(history.step,history.loss,alpha=.35,label='mini-batch')
    if len(history)>10:
        smooth=history.loss.rolling(min(50,len(history)),min_periods=1).mean()
        ax.plot(history.step,smooth,label='rolling mean')
    ax.set_title('Actual SimCLR training loss (NOT fish length MAE)')
    ax.set_xlabel('Optimizer step');ax.set_ylabel('InfoNCE loss');ax.legend()
    plt.tight_layout();plt.savefig(OUT/'self_supervised_training_loss.png',dpi=160);plt.show()

# Portable inference-only encoder uses ordinary PyTorch state_dict.
torch.save({'encoder_state_dict':model.state_dict(),
            'image_size':IMAGE_SIZE,'embedding_dim':128,
            'task':'self_supervised_image_embeddings_not_metric_length',
            'source_bags':len(bags),'completed_bags':complete_count,
            'trained_images':int(coverage.trained_images.sum())},OUT/'visual_encoder_weights.pt')

report={'task':'self_supervised_visual_pretraining','fish_length_regression':'NOT TRAINED',
        'source_bags':len(bags),'fully_traversed_bags':complete_count,
        'decoded_images':int(coverage.decoded_images.sum()),
        'trained_images':int(coverage.trained_images.sum()),
        'decode_failures':int(coverage.decode_failures.sum()),
        'optimizer_steps':int(state['global_step']),
        'full_available_dataset_processed':bool(complete_count==len(bags) and coverage.decode_failures.sum()==0 and FRAME_STRIDE==1 and MAX_FRAMES_PER_BAG is None and MAX_TRAIN_STEPS is None),
        'missing_source_recording':'Dataset B source archive truncated: unavailable recordings not trainable',
        'limitations':['No validated per-frame fish length labels for all frames',
                       'Not a centimetre prediction model',
                       'All Dataset A available bags from one fish identity; split by frames would leak',
                       'ROS stereo geometry and synchronization not yet validated']}
write_json_atomic(OUT/'training_report.json',report)
(OUT/'README.md').write_text(' '.join(['FishSizeEstimate','Project','streaming','visual','pretraining.','Consult','training_report.json','and','all_bag_coverage.csv.','Weights','are','NOT','a','centimetre','prediction','model.']),encoding='utf-8')
print('Training report:',json.dumps(report,indent=2))
print('Output artifacts:')
for p in sorted(OUT.iterdir()):
    if p.is_file():print(' ',p.name,round(p.stat().st_size/1024**2,3),'MiB')

## 7 · Stereo stream timing QA — independent of model loss

This uses the ROS **bag receive timestamps** saved during streaming for each camera. It calculates one-to-one, tolerance-bounded proximity between left and right image streams, without revisiting the giant recordings. It does *not* establish matching scene content, capture-time clock alignment, rectification or 3-D scale. `STEREO_MAX_DELTA_MS` is an exploratory threshold, not validated hardware tolerance.

If timestamp arrays are absent because a checkpoint came from an older notebook, the result is explicitly `NOT_MEASURED`.


In [ ]:
def match_stereo_timestamps(left_ns, right_ns, max_delta_ms):
    # Greedy one-to-one matching with monotonically ordered timestamps.
    left = sorted(int(x) for x in left_ns)
    right = sorted(int(x) for x in right_ns)
    threshold = int(max_delta_ms * 1e6)
    i = j = 0
    errors_ms = []
    while i < len(left) and j < len(right):
        d = left[i] - right[j]
        if abs(d) <= threshold:
            errors_ms.append(abs(d) / 1e6)
            i += 1
            j += 1
        elif d < 0:
            i += 1
        else:
            j += 1
    return {
        'paired': len(errors_ms),
        'median_delta_ms': float(np.median(errors_ms)) if errors_ms else None,
        'p95_delta_ms': float(np.percentile(errors_ms, 95)) if errors_ms else None,
        'max_delta_ms': float(max(errors_ms)) if errors_ms else None,
    }

qa_rows = []
for bag in bags:
    s = state['bag_stats'].get(bag.name, {})
    ts = s.get('camera_timestamps_ns')
    counts = s.get('camera_image_counts', {})
    if ts is None or not ts.get('left') or not ts.get('right'):
        result = dict(paired=0, median_delta_ms=None, p95_delta_ms=None,
                      max_delta_ms=None)
        status = 'NOT_MEASURED'
    else:
        result = match_stereo_timestamps(ts['left'], ts['right'], STEREO_MAX_DELTA_MS)
        status = 'MEASURED' if result['paired'] else 'NO_PAIRS_WITHIN_TOLERANCE'
    qa_rows.append({
        'bag': bag.name, 'status': status,
        'left_images': int(counts.get('left', 0)),
        'right_images': int(counts.get('right', 0)),
        'time_threshold_ms': STEREO_MAX_DELTA_MS,
        **result
    })
stereo_df = pd.DataFrame(qa_rows)
display(stereo_df)
stereo_df.to_csv(OUT/'stereo_timestamp_qa.csv', index=False)
if (stereo_df.status == 'MEASURED').any():
    measured = stereo_df[stereo_df.status == 'MEASURED']
    fig, ax = plt.subplots(figsize=(12,4))
    ax.bar(range(len(measured)), measured['p95_delta_ms'].astype(float))
    ax.set_xticks(range(len(measured)), measured['bag'].str.slice(0,12), rotation=45, ha='right')
    ax.set_ylabel('95th percentile bag timestamp difference (ms)')
    ax.set_title('Left/right nearest timestamp diagnostic — not stereo calibration')
    plt.tight_layout()
    plt.savefig(OUT/'stereo_timestamp_p95.png',dpi=150)
    plt.show()
print('Timestamp QA completed; no claim of true camera-exposure synchronization.')

## 8 · Calibration matrix inspection & scale-readiness

YAML camera intrinsics/extrinsics belong to distinct sessions. This section resolves original paths via `source_manifest.json`, prints camera matrix/projection fields and notes candidate stereo baseline **only as a mathematical candidate**. The units of translation and validity of rectification are not verified. **Do not output cm predictions from an inferred baseline.**


In [ ]:
try:
    import yaml
except ImportError:
    print('PyYAML unavailable — install with %pip install pyyaml if Internet enabled')
    yaml = None

cal_rows = []
for p in all_files:
    if p.suffix.lower() not in ('.yaml','.yml'):
        continue
    orig = original_path(p)
    row = {'original_path':orig, 'parsed':False, 'fx_px':None, 'fy_px':None,
           'P03_over_fx_candidate':None,'warning':None}
    if yaml is None:
        row['warning']='PyYAML missing'
    else:
        try:
            d = yaml.safe_load(p.read_text(encoding='utf-8-sig'))
            if not isinstance(d, dict):
                raise ValueError('Expected YAML mapping')
            def get_matrix(key, expected):
                v=d.get(key)
                if isinstance(v,dict):v=v.get('data')
                if not isinstance(v,(tuple,list)) or len(v)!=expected:return None
                return np.array(v,dtype=float)
            K=get_matrix('camera_matrix',9)
            P=get_matrix('projection_matrix',12)
            if K is not None:
                row['fx_px']=float(K[0]); row['fy_px']=float(K[4])
            if P is not None and float(P[0])!=0:
                row['P03_over_fx_candidate']=float(-P[3]/P[0])
            row['parsed']=True
        except Exception as ex:
            row['warning']=f'{type(ex).__name__}: {str(ex)[:140]}'
    cal_rows.append(row)
calibration_df=pd.DataFrame(cal_rows)
display(calibration_df)
calibration_df.to_csv(OUT/'calibration_qa.csv', index=False)
print('WARNING: candidate translation values have UNKNOWN validated physical units.')
print('Camera matrix alone is insufficient to determine fish total length (cm).')

## 9 · Learned encoder sanity check (NOT held-out performance)

Runs inference on only the actual frames already decoded during preview. The notebook verifies the encoder produces finite, normalized embeddings. Because those views may have appeared during training, this is an **in-sample engineering check**, not an independent evaluation and not a claim about centimetre estimation accuracy.


In [ ]:
model.eval()
embeddings=[]
with torch.no_grad():
    for topic, rgb in preview:
        rgb_small=cv2.resize(rgb,(IMAGE_SIZE,IMAGE_SIZE),interpolation=cv2.INTER_AREA)
        x=torch.from_numpy(np.ascontiguousarray(rgb_small)).permute(2,0,1).unsqueeze(0).float().to(DEVICE)/255.0
        feature,_=model(x)
        embeddings.append(feature.squeeze(0).float().cpu().numpy())
assert embeddings, 'No preview embeddings'
embeddings=np.stack(embeddings)
assert np.isfinite(embeddings).all(), 'Non-finite encoder embeddings'
norms=np.linalg.norm(embeddings,axis=1)
print('Preview embeddings:',embeddings.shape)
print('L2 norms:',norms.round(5))
print('Finite:',bool(np.isfinite(embeddings).all()))
cos_sim=embeddings @ embeddings.T
fig,ax=plt.subplots(figsize=(6,5))
mat=ax.imshow(cos_sim,vmin=-1,vmax=1,cmap='coolwarm')
ax.set_title('Visual embedding similarity (in-sample sanity check)')
ax.set_xticks(range(len(preview)),[f'F{i+1}' for i in range(len(preview))])
ax.set_yticks(range(len(preview)),[f'F{i+1}' for i in range(len(preview))])
fig.colorbar(mat,ax=ax,label='cosine similarity')
plt.tight_layout()
plt.savefig(OUT/'embedding_similarity_preview.png',dpi=160)
plt.show()

## 10 · Evidence-based final report / scientific readiness

No synthetic validation numbers. V6 requires *genuinely held-out fish identities or recording sessions*, verified per-frame fish identity, temporally aligned left/right images, rectification and known camera baseline units to evaluate length. **A train/validation split by adjacent frames would leak near-duplicate views.** The 11 bags are the available recovered subset, not the entire original Zenodo release.

**Data split for a future supervised cm experiment:** explicit fish-identity grouping, with matched pairs and endpoint annotations. For Dataset B multi-fish frames, track IDs are only usable after checking the mapping JSON against visual tracks. Independent fish count is too small here for trustworthy broad generalization claims.


In [ ]:
readiness = {
    'all_11_available_bags_stream_completed': bool(complete_count == len(bags)),
    'image_decode_zero_failures': bool(int(coverage.decode_failures.sum()) == 0),
    'actual_optimizer_updates': bool(int(state['global_step']) > 0),
    'stereo_bag_timestamp_proximity_measured_for_every_bag': bool((stereo_df.status == 'MEASURED').all()),
    'physical_baseline_units_verified': False,
    'camera_rectification_verified': False,
    'fish_3D_nose_tail_points_annotated_and_verified': False,
    'independent_fish_id_heldout_cm_evaluation': False,
}
report_v6 = {
    'project': 'FishSizeEstimate',
    'pipeline': 'full available ROS-frame self-supervised pretraining + stereo evidence',
    'source_bag_count': int(len(bags)),
    'completed_bags': int(complete_count),
    'train_frames': int(coverage.trained_images.sum()),
    'optimizer_steps': int(state['global_step']),
    'stereo_timestamp_diagnostic_only': True,
    'cm_length_model_trained': False,
    'length_mae_cm': None,
    'length_rmse_cm': None,
    'model_evaluation_status': 'NOT EVALUATED ON INDEPENDENT FISH LENGTH LABELS',
    'source_dataset_status': manifest.get('status'),
    'readiness': readiness,
    'limitations': [
        'Only 11 available recovered ROS bags were processed, not missing upstream recordings',
        'Available Dataset A recordings all correspond to Fish 0 (one length label)',
        'Two available Dataset B recordings are multi-fish',
        'Timestamp proximity alone is not synchronized stereo geometry',
        'No statistically defensible held-out cm estimation evaluation yet',
        'Check original dataset, checkpoint and photograph licenses before publishing',
    ],
}
(OUT/'research_readiness_v6.json').write_text(
    json.dumps(report_v6,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
readiness_df = pd.DataFrame([{'criterion':k,'verified':v} for k,v in readiness.items()])
display(readiness_df)
readiness_df.to_csv(OUT/'readiness_v6.csv',index=False)
print('OUTPUTS:')
for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(f'{p.name:42} {p.stat().st_size/(1024**2):8.3f} MiB')
print('\nCOMPLETE:',complete_count,'/',len(bags),'| steps:',state['global_step'])
print('RESEARCH STATUS: visual representation trained; metric length estimator NOT YET VALIDATED.')

## 11 · Ground-truth / recording identity join (auditable)

Use the source `Dataset_Summary_Dataset_A.csv` and `Dataset_Summary_Dataset_B.csv`, **not filename guesses**, to link available bag files to reported specimen identities. This is a **recording-level** mapping. Multi-fish recordings remain ambiguous until visual track identity is verified. Only known single-fish recordings are eligible for subject-level label assignment, but there is currently just **one independent Fish 0 identity** in uploaded Dataset A bags.


In [ ]:
import re
summary_frames=[]
for name in ('Dataset_Summary_Dataset_A.csv','Dataset_Summary_Dataset_B.csv'):
    p=next((f for f in all_files if f.name.endswith(name)),None)
    if p is None: continue
    t=pd.read_csv(p,dtype=str,encoding='utf-8-sig').fillna('')
    t['summary_source']=name
    summary_frames.append(t)
summary=pd.concat(summary_frames,ignore_index=True) if summary_frames else pd.DataFrame()
assert len(summary)>0, 'Recording summary missing'

def clean_basename(v):
    return Path(str(v).replace('\\','/')).name

def locate_summary(bag):
    basename=clean_basename(original_path(bag))
    sub=summary[summary['Bagfile_Name'].map(clean_basename)==basename]
    if len(sub)==1:return sub.iloc[0].to_dict()
    if len(sub)>1:return {'ambiguous_matches':len(sub)}
    return {}

join_rows=[]
for bag in bags:
    rec=locate_summary(bag)
    join_rows.append({'bag':bag.name,'source_path':original_path(bag),
                      'summary_match':bool(rec) and 'ambiguous_matches' not in rec,
                      'dataset_type':rec.get('Dataset_Type'),
                      'fishes_present':rec.get('Fishes_Present'),
                      'date':rec.get('Date'), 'summary_ambiguity':rec.get('ambiguous_matches')})
recording_labels=pd.DataFrame(join_rows)
recording_labels.to_csv(OUT/'recording_ground_truth_join.csv',index=False)
display(recording_labels)
print('Matched:',int(recording_labels.summary_match.sum()),'/',len(bags))
print('CAUTION: A multi-fish recording is never a single-fish regression label.')

## 12 · Preview real left/right stereo pairs from **every available ROS recording**

The training pass already consumed **all image messages**; this section revisits each recording only long enough to find the **first matched real image pair**. Pairing uses ROS-bag receive timestamps within an *exploratory 50 ms tolerance*. It does **not** prove exact exposure synchronization. We export a JPEG comparison for each recording, plus the pair timestamps, topics and pixel dimensions. A pair is kept separate from its GT until identity is verifiable.


In [ ]:
PAIR_LIMIT_NS=int(STEREO_MAX_DELTA_MS*1e6)
PAIR_PREVIEW_EDGE=840
pair_rows=[]
paired_previews=[]

def shrink_frame(img,edge=PAIR_PREVIEW_EDGE):
    h,w=img.shape[:2]
    scale=min(1.0,edge/max(h,w))
    return cv2.resize(img,(max(1,int(round(w*scale))),max(1,int(round(h*scale))))) if scale<1 else img.copy()

def capture_first_pair(bag):
    latest={'left':None,'right':None}
    with AnyReader([bag]) as reader:
        conns=[c for c in reader.connections if is_image_topic(c) and
               ('/left/' in c.topic.lower() or '/right/' in c.topic.lower())]
        for count,(conn,ts,raw) in enumerate(reader.messages(connections=conns)):
            if count>300:break
            side='left' if '/left/' in conn.topic.lower() else 'right'
            try:
                msg=reader.deserialize(raw,conn.msgtype)
                img=decode_image(msg,conn.msgtype)
                if img is None:continue
            except Exception:continue
            latest[side]=(int(ts),conn.topic,img)
            other='right' if side=='left' else 'left'
            if latest[other] and abs(latest[side][0]-latest[other][0])<=PAIR_LIMIT_NS:
                return latest['left'],latest['right']
    return None

for idx,bag in enumerate(bags):
    try:
        pair=capture_first_pair(bag)
        if pair is None:
            pair_rows.append({'bag':bag.name,'status':'NO_PAIR','delta_ms':None})
            print('[NO PAIR]',idx,bag.name)
            continue
        l,r=pair
        lsmall,rsmall=shrink_frame(l[2]),shrink_frame(r[2])
        # same height for side-by-side comparison
        h=min(lsmall.shape[0],rsmall.shape[0])
        lsmall=cv2.resize(lsmall,(round(lsmall.shape[1]*h/lsmall.shape[0]),h))
        rsmall=cv2.resize(rsmall,(round(rsmall.shape[1]*h/rsmall.shape[0]),h))
        combined=np.concatenate([lsmall,rsmall],axis=1)
        outname=f'stereo_pair_{idx:02d}.jpg'
        assert cv2.imwrite(str(OUT/outname),combined)
        pair_rows.append({'bag':bag.name,'status':'PAIR_FOUND','delta_ms':abs(l[0]-r[0])/1e6,
                          'left_topic':l[1],'right_topic':r[1],
                          'left_h':l[2].shape[0],'left_w':l[2].shape[1],
                          'right_h':r[2].shape[0],'right_w':r[2].shape[1],
                          'preview':outname})
        paired_previews.append({'bag':bag.name,'left':l[2],'right':r[2],'delta_ms':abs(l[0]-r[0])/1e6})
        print('[PAIR]',idx,'delta_ms=',round(abs(l[0]-r[0])/1e6,3),outname)
    except Exception as ex:
        pair_rows.append({'bag':bag.name,'status':'ERROR','error':str(ex)[:240]})
        print('[PAIR ERROR]',bag.name,type(ex).__name__,str(ex)[:180])
pairs_df=pd.DataFrame(pair_rows)
pairs_df.to_csv(OUT/'real_stereo_pair_inventory.csv',index=False)
display(pairs_df)
print('Pairs:',len(paired_previews),'/',len(bags))

## 13 · Session-aware calibration pairs and baseline candidates

Match original YAML paths by their source date/dataset grouping, and use *both left and right* from the same calibration session. Do not confuse camera `K` intrinsics with stereo extrinsics. Compute projection baseline candidate `-P_right[0,3]/P_right[0,0]` only for inspection, and **never silently assume metres**. This is a diagnostic table, not accepted physical calibration.


In [ ]:
cal_dict={}
if yaml is not None:
    for f in all_files:
        if f.suffix.lower() not in ('.yaml','.yml'):continue
        orig=original_path(f)
        try:
            doc=yaml.safe_load(f.read_text(encoding='utf-8-sig'))
        except Exception as ex:
            print('Calibration parse failure:',orig,ex);continue
        if not isinstance(doc,dict):continue
        side=Path(orig).stem.lower()
        session=str(Path(orig).parent)
        cal_dict.setdefault(session,{})[side]=doc

def matrix_from(doc,key,shape):
    v=doc.get(key)
    if isinstance(v,dict):v=v.get('data')
    if not isinstance(v,(list,tuple)):return None
    arr=np.array(v,dtype=np.float64)
    return arr.reshape(shape) if arr.size==np.prod(shape) else None

cal_pair_rows=[]
for session,doc in sorted(cal_dict.items()):
    ld,rd=doc.get('left'),doc.get('right')
    if not ld or not rd:
        cal_pair_rows.append({'session':session,'has_pair':False});continue
    pl=matrix_from(ld,'projection_matrix',(3,4))
    pr=matrix_from(rd,'projection_matrix',(3,4))
    kl=matrix_from(ld,'camera_matrix',(3,3))
    kr=matrix_from(rd,'camera_matrix',(3,3))
    b=(-pr[0,3]/pr[0,0]) if pr is not None and abs(pr[0,0])>0 else np.nan
    cal_pair_rows.append({'session':session,'has_pair':True,'left_K_valid':kl is not None,
        'right_K_valid':kr is not None,'projections_valid':pl is not None and pr is not None,
        'baseline_candidate_UNVERIFIED_UNITS':b,
        'width':ld.get('image_width'),'height':ld.get('image_height')})
cal_pairs_df=pd.DataFrame(cal_pair_rows)
cal_pairs_df.to_csv(OUT/'calibration_stereo_pair_candidates.csv',index=False)
display(cal_pairs_df)
print('These candidates MUST NOT be treated as physical length units without source validation.')

## 14 · Stereo disparity preview (real frames; qualitative diagnostics)

`StereoSGBM` compares the left/right images from a real recorded pair. This code first checks matching dimensions and uses the supplied **rectification maps only when available and verified**. Without verified rectification/extrinsics, the output is a **raw correspondence diagnostic only**, not depth or fish length. This section does not pretend fish segmentation, nose/tail endpoints, or millimetres have been established.


In [ ]:
# Inspect a single accessible pair for correspondence; no per-fish length prediction.
if paired_previews:
    item=paired_previews[0]
    left,right=item['left'],item['right']
    if left.shape[:2]!=right.shape[:2]:
        print('SKIP disparity: left/right image dimensions differ')
    else:
        # Keep aspect ratio; block matching is computationally bounded.
        max_w=960
        h,w=left.shape[:2]
        scale=min(1.0,max_w/w)
        W=max(128,int(w*scale)//16*16)
        H=max(96,int(h*W/w))
        L=cv2.cvtColor(cv2.resize(left,(W,H)),cv2.COLOR_BGR2GRAY)
        R=cv2.cvtColor(cv2.resize(right,(W,H)),cv2.COLOR_BGR2GRAY)
        stereo=cv2.StereoSGBM_create(minDisparity=0,numDisparities=128,blockSize=7,
            P1=8*7*7,P2=32*7*7,disp12MaxDiff=2,uniquenessRatio=10,
            speckleWindowSize=100,speckleRange=2)
        raw_disp=stereo.compute(L,R).astype('float32')/16.0
        good=(raw_disp>0)&np.isfinite(raw_disp)
        print('Valid raw disparity pixel fraction:',round(float(good.mean()),4))
        fig,axs=plt.subplots(1,3,figsize=(16,5),constrained_layout=True)
        axs[0].imshow(cv2.cvtColor(L,cv2.COLOR_GRAY2RGB));axs[0].set_title('Actual left camera')
        axs[1].imshow(cv2.cvtColor(R,cv2.COLOR_GRAY2RGB));axs[1].set_title('Actual right camera')
        arr=np.ma.masked_where(~good,raw_disp)
        axs[2].imshow(arr,cmap='viridis',vmin=0,vmax=128)
        axs[2].set_title('Unrectified disparity diagnostic — NOT metric depth')
        for ax in axs:ax.axis('off')
        plt.savefig(OUT/'stereo_raw_disparity_diagnostic.png',dpi=130,bbox_inches='tight')
        plt.show()
else:
    print('SKIP disparity: no verified-decodable stereo candidate pairs')

## 15 · Endpoint annotations: create a genuine supervision contract

**Non-negotiable scientific gate.** To estimate length you must identify snout/tail tips in *both rectified camera images* for a known fish and an agreed baseline unit. The notebook writes a template rather than fabricating endpoint positions. For multi-fish Dataset B, link the physical fish identity using the provided track annotation and manual review. Capture-time/stereo geometry is not validated by bag receive timestamps alone.

The template can be filled outside the notebook and attached as a second Kaggle input. When genuine measurements are available, the following optional block calculates 3D point-to-point length and scores the results. **Without verified labels the block skips and exports `NOT_EVALUATED`.**


In [ ]:
cols=['bag','timestamp_ns','fish_id','dataset','left_nose_x','left_nose_y',
      'right_nose_x','right_nose_y','left_tail_x','left_tail_y',
      'right_tail_x','right_tail_y','calibration_session','baseline_m_verified',
      'rectified_verified','identity_verified','endpoint_verified','ground_truth_cm']
template=OUT/'fish_endpoint_annotations_TEMPLATE.csv'
if not template.exists(): pd.DataFrame(columns=cols).to_csv(template,index=False)
print('Annotation template:',template)
print('Rows intentionally empty: no invented keypoints or calibrated lengths.')
print('Annotate coordinates on ORIGINAL full-resolution frames, not resized previews.')
print('Each nose/tail pixel must be checked in LEFT and RIGHT views.')

## 16 · Strict gated 3D metric calculation and independent evaluation

The following implementation is **optional** and is activated only if an annotated CSV is attached (its default path is shown). It refuses incomplete evidence: verified baseline in **metres**, verified rectification, verified fish identity, verified endpoint correspondences, matching left/right 2D points, positive disparity, GT from the original nine fish, and **held-out identity separation**. It computes three-dimensional endpoints with `Z=fx*B/d`, then Euclidean snout-to-tail distance. This is a simplified straight-line total-length approximation, not a validated solution for curved fish. A proper evaluation additionally needs physically correct distortion/rectification and camera extrinsics.

The notebook **will not automatically mark geometry or labels verified** just because CSV columns say so. The booleans represent assertions supplied by the independent human annotator; all numeric measurements remain candidates pending external review. Training a regression head on 1–3 independent fish identities is scientifically weak, so no arbitrary deep regression metrics are generated.


In [ ]:
ANNOTATIONS_CSV=Path('/kaggle/input/fishsizeestimate-verified-annotations/fish_endpoint_annotations.csv')
metric_rows=[]
metric_status='NOT_EVALUATED_NO_VERIFIED_ANNOTATIONS'

def as_true(x):return str(x).strip().lower() in ('true','1','yes','y')

def triangulate_rectified(xl,yl,xr,yr,fx,fy,cx,cy,baseline_m):
    disparity=xl-xr
    if not np.isfinite(disparity) or disparity<=0:return None
    z=fx*baseline_m/disparity
    return np.array([(xl-cx)*z/fx,(yl-cy)*z/fy,z],dtype=float)

if ANNOTATIONS_CSV.is_file():
    ann=pd.read_csv(ANNOTATIONS_CSV)
    missing=set(cols)-set(ann.columns)
    if missing: raise ValueError(f'Missing annotation columns: {missing}')
    # Only independent human-verified annotations; do not auto-label or impute.
    for row in ann.to_dict('records'):
        if not (as_true(row['rectified_verified']) and as_true(row['identity_verified']) and
                as_true(row['endpoint_verified'])):continue
        match=gt[(gt['Identifier']==str(row['fish_id'])) & (gt['Dataset']==str(row['dataset']))]
        if len(match)!=1:continue
        true_cm=float(match.iloc[0]['Total_Length_cm'])
        if abs(float(row['ground_truth_cm'])-true_cm)>1e-6:continue
        sess=str(row['calibration_session'])
        if sess not in cal_dict:continue
        cd=cal_dict[sess].get('left')
        if not isinstance(cd,dict):continue
        K=matrix_from(cd,'camera_matrix',(3,3))
        if K is None:continue
        B=float(row['baseline_m_verified'])
        if not np.isfinite(B) or B<=0:continue
        fx,fy,cx,cy=float(K[0,0]),float(K[1,1]),float(K[0,2]),float(K[1,2])
        # We assume that the manually provided coordinates are in precisely this
        # calibrated, rectified camera geometry; requires independent verification.
        try:
            nose=triangulate_rectified(*(float(row[v]) for v in ['left_nose_x','left_nose_y','right_nose_x','right_nose_y']),fx,fy,cx,cy,B)
            tail=triangulate_rectified(*(float(row[v]) for v in ['left_tail_x','left_tail_y','right_tail_x','right_tail_y']),fx,fy,cx,cy,B)
            if nose is None or tail is None:continue
            estimate_cm=float(np.linalg.norm(nose-tail)*100)
            if not np.isfinite(estimate_cm) or estimate_cm<=0:continue
            metric_rows.append({'bag':row['bag'],'fish_id':row['fish_id'],'dataset':row['dataset'],
                'gt_cm':true_cm,'candidate_length_cm':estimate_cm,
                'absolute_error_cm':abs(estimate_cm-true_cm)})
        except (ValueError,TypeError,OverflowError):continue
else:
    print('[METRIC SKIPPED] No external verified annotation CSV attached.')
if metric_rows:
    candidate_df=pd.DataFrame(metric_rows)
    candidate_df.to_csv(OUT/'candidate_metric_lengths_REVIEW_REQUIRED.csv',index=False)
    display(candidate_df)
    unique_ids=candidate_df[['dataset','fish_id']].drop_duplicates()
    print('Independent annotated fish identities:',len(unique_ids))
    metric_status='CANDIDATES_REQUIRE_EXTERNAL_GEOMETRY_VALIDATION'
    if len(unique_ids)<3:
        print('Insufficient unique identities for meaningful held-out evaluation; MAE/RMSE intentionally suppressed.')
    else:
        print('Candidate geometric errors available, but still not independently held-out test performance.')
else:
    print('No admissible 3D measurements. No fake length MAE/RMSE reported.')
print('Metric status:',metric_status)

## 17 · Final deliverables, limits, and machine-readable research report

Outputs include actual trained visual encoder, its loss curve, all-bag coverage, camera QA, pairing previews from real ROS messages, a qualitative disparity map if possible, calibration candidates, an explicit blank endpoint template, and a final experimental readiness document. This represents the **complete available-data visual pretraining + stereo research workflow**, *not a final validated fish-size product*.


In [ ]:
final_status={
 'project':'FishSizeEstimate',
 'available_ros_bags':len(bags),
 'completed_training_bags':int(complete_count),
 'training_image_messages':int(coverage.trained_images.sum()),
 'optimizer_steps':int(state['global_step']),
 'decoded_failures':int(coverage.decode_failures.sum()),
 'real_stereo_pairs_previewed':int(len(paired_previews)),
 'stereo_timestamp_qa_bags':int((stereo_df.status=='MEASURED').sum()),
 'calibration_pair_candidates':int(cal_pairs_df.has_pair.fillna(False).sum()) if 'has_pair' in cal_pairs_df else 0,
 'independent_confirmed_length_regressor':False,
 'metric_evaluation_status':metric_status,
 'metric_mae_cm':None,
 'metric_rmse_cm':None,
 'dataset_recovery':manifest.get('status'),
 'limitations':manifest.get('limitations',[])+[
   'All recovered Dataset A recordings correspond to Fish 0 only',
   'Dataset B requires per-frame multi-fish association',
   'Stereo receive-time matching is not proof of exposure synchronization',
   'Calibrated metric stereo and 3D nose/tail measurements not independently verified',
   'A representation encoder is not a fish-size regressor']
}
(OUT/'integrated_research_report.json').write_text(json.dumps(final_status,indent=2,ensure_ascii=False),encoding='utf-8')
readme_txt="""# FishSizeEstimate — integrated research outputs

This experiment uses **all available recovered ROS recordings** for self-supervised visual pretraining and creates **real** stereo QA / disparity diagnostic artifacts. It **does not claim a calibrated centimetre estimator**. See `integrated_research_report.json` for coverage and open scientific gates.

Important files:
- `visual_encoder_weights.pt`, `latest_checkpoint.pt`: learned encoder & training state
- `train_steps.csv`, `self_supervised_training_loss.png`: optimization evidence
- `all_bag_coverage.csv`: full available recording coverage
- `real_stereo_pair_inventory.csv`, `stereo_pair_XX.jpg`: actual stereo preview pairs
- `stereo_timestamp_qa.csv`, `calibration_stereo_pair_candidates.csv`: camera QA
- `stereo_raw_disparity_diagnostic.png`: unrectified **non-metric** disparity preview (when possible)
- `fish_endpoint_annotations_TEMPLATE.csv`: required manual stereo keypoints, empty on purpose
- `integrated_research_report.json`: honest completion/readiness report

Scientific limitations: nine source length values but recovered A videos all feature one fish; Dataset B incompletely recovered; identity/geometry validation and independent held-out cm performance remain open. Do not redistribute third-party data or checkpoints without checking rights.
"""
(OUT/'README.md').write_text(readme_txt,encoding='utf-8')
print(json.dumps(final_status,indent=2,ensure_ascii=False))
print('\nARTIFACTS')
for p in sorted(OUT.iterdir()):
    if p.is_file():print(p.name,round(p.stat().st_size/(1024**2),3),'MiB')
print('FULL-RECORDING VISUAL TRAINING:', 'PASS' if complete_count==len(bags) else 'PARTIAL')
print('FISH LENGTH (cm) VALIDATION:',metric_status)

## 18 · GitHub publication checklist

- Publish the clean notebook, reports, QA previews and code only if permitted by the original source licenses.
- Cite the Kaggle data page: `https://www.kaggle.com/datasets/wafabila/fishsizeestimate-dataset` and the original Zenodo dataset after DOI/license verification.
- Clearly label **self-supervised pretraining** (10,405 images / 11 of 11 available recovered bags, according to V6 observed log) separately from **fish total length (cm)** (NOT VALIDATED).
- Never publish raw ~23 GiB ROS data or a third-party segmentation checkpoint in Git without permission.
- Do not use the nine individual fish as though they were thousands of independent physical measurements.
- When independent verified endpoint/stereo annotations exist, attach the annotation dataset and use the gated cell to generate *candidate* lengths; report actual independent evaluation only after a defensible subject-level held-out protocol.
